# Train model 

In [1]:
import cv2 as cv
import os
import mediapipe as mp 
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
import joblib

In [2]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
recognizer = cv.face.LBPHFaceRecognizer_create() 

In [3]:
database = 'Dataset' 

walk_result = list(os.walk(database))
dataY = walk_result[0][1]

folder = []
for item in walk_result : 
    folder_path = item[0]
    folder.append(folder_path) 

folder = folder[1:]
data = []
count = 0

for path in folder : 
    files = os.listdir(path) 
    label = os.path.split(path)[-1]

    for file in files :
        full_path = os.path.join(path, file)

        # conver to gray scale 
        image = cv.imread(full_path) 
        img_rgb = cv.cvtColor(image, cv.COLOR_BGR2RGB) 

        with mp_hands.Hands(static_image_mode=True, max_num_hands=1) as hands : 
            result = hands.process(img_rgb)
            
            if result.multi_hand_landmarks :                 
                for hand_landmarks in result.multi_hand_landmarks : 
                    row =[]

                    for lm in hand_landmarks.landmark : 
                        row.append(lm.x)
                        row.append(lm.y)

                    row.append(label) 
                    data.append(row)
    print(f'{label} processing completed')
    count += 1
    if count == 15 : 
        break
print(data)

# Create columns and name 
columns = []
for i in range(21):
    columns += [f"x{i}", f"y{i}"]
columns.append("label")

df = pd.DataFrame(data=data, columns=columns) 

# Save to CSV 
os.makedirs('Model', exist_ok=True)
df.to_csv('Model/asl_data.csv', index=False) 
print('CSV Created as asl_data.csv')

0 processing completed
1 processing completed
2 processing completed
3 processing completed
4 processing completed
5 processing completed
6 processing completed
7 processing completed
8 processing completed
9 processing completed
a processing completed
b processing completed
c processing completed
d processing completed
e processing completed
[[0.30983632802963257, 0.33478423953056335, 0.3416069746017456, 0.32744109630584717, 0.38675516843795776, 0.29366809129714966, 0.4496811032295227, 0.2790496349334717, 0.5069965124130249, 0.2807992696762085, 0.3349142074584961, 0.1004122644662857, 0.5087324976921082, 0.12465163320302963, 0.6005590558052063, 0.1767587661743164, 0.6557902097702026, 0.2155171036720276, 0.33649584650993347, 0.051119767129421234, 0.5344398021697998, 0.06560385972261429, 0.6274455785751343, 0.125442236661911, 0.6849867701530457, 0.17853116989135742, 0.35661014914512634, 0.041266053915023804, 0.5446895360946655, 0.052074261009693146, 0.6342929005622864, 0.1131460070610046

In [4]:
# Load Data 
data = pd.read_csv('./Model/asl_data.csv')

# Encode the labels 
encoder = LabelEncoder() 
data['label'] = encoder.fit_transform(data['label'])

# Split features and target
X = data.drop('label', axis=1)
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=42 
)

# Train using SVC Model 
model = SVC(kernel='rbf', probability=True) 
model.fit(X_train, y_train) 
print('Model is trained') 

# Check accuracy 
accuracy = model.score(X_test, y_test) 
print(f'Accuracy: {accuracy * 100}%') 

# Save the model 
joblib.dump(model, 'Model/asl_model.pkl')
joblib.dump(encoder, 'Model/label_encoder.pkl')

print('Model and encoder saved!')

Model is trained
Accuracy: 97.48427672955975%
Model and encoder saved!
